
# Vanilla RAG on Legal Documents

**Day 3 — RAG & Agents · Practical 1 of 6 · Companion to the "RAG Fundamentals" deck**

> **Running in Google Colab:** works fine on the default **CPU runtime** — no GPU needed.

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. Chunk a small legal-document corpus and embed it into a vector store
2. Query that vector store end-to-end, from a question to a grounded LLM answer
3. See the deck's two-phase architecture (Ingestion, Query) as real, runnable code

## Why This Matters for a Law Firm

This is the baseline every later Day 3 notebook builds on. Get this pipeline working end-to-end
first — hybrid retrieval, reranking, self-correction, and multimodal retrieval (later
notebooks) are all refinements ON TOP of this same skeleton, not replacements for it.

## Notebook Workflow

```mermaid
flowchart LR
    A["Legal document\ncorpus"] --> B["Chunking"]
    B --> C["Embedding model"]
    C --> D["ChromaDB\n(local, in-notebook)"]
    E["User question"] --> F["Embed question"]
    F --> D
    D --> G["Top-k chunks"]
    G --> H["LLM prompt\n+ retrieved context"]
    H --> I["Grounded answer"]



## Section 1 — Setup

**ChromaDB in local/in-memory mode** needs no external server or account — it runs entirely
inside this notebook's process, making it the simplest possible vector store for a Colab demo.


In [ ]:

%pip install -q chromadb sentence-transformers openai

import chromadb
from sentence_transformers import SentenceTransformer

print("ChromaDB and sentence-transformers ready.")



## Section 2 — API Key Setup

We use an LLM API for the final generation step. Store your key in Colab Secrets (🔑 icon in
the left sidebar) rather than hard-coding it.


In [ ]:

import os

def get_api_key(env_var_name):
    try:
        from google.colab import userdata
        key = userdata.get(env_var_name)
        if key:
            return key
    except ImportError:
        pass
    return os.environ.get(env_var_name)

OPENAI_API_KEY = get_api_key("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError(
        "No OPENAI_API_KEY found. In Colab: add a secret named OPENAI_API_KEY via the key "
        "icon in the left sidebar. Locally: set the OPENAI_API_KEY environment variable."
    )

from openai import OpenAI
client = OpenAI(api_key=OPENAI_API_KEY)
print("OpenAI client configured.")



## Section 3 — A Small Legal Document Corpus

A handful of short contract clauses, standing in for a real document corpus. In production this
would be your firm's actual precedent bank or a client's document set (loaded and chunked from
real files) -- kept small and self-contained here so the notebook runs instantly.


In [ ]:

legal_documents = [
    {
        "id": "doc1_indemnification",
        "text": (
            "Section 7.1 Indemnification. The Contractor shall indemnify, defend, and hold "
            "harmless the Client from and against any and all claims, damages, losses, and "
            "expenses, including reasonable attorneys' fees, arising out of or resulting from "
            "the Contractor's gross negligence or willful misconduct in the performance of "
            "this Agreement."
        ),
    },
    {
        "id": "doc2_termination",
        "text": (
            "Section 9.2 Termination for Convenience. Either party may terminate this "
            "Agreement without cause upon sixty (60) days' prior written notice to the other "
            "party. Upon termination, the Client shall pay the Contractor for all services "
            "performed through the effective date of termination."
        ),
    },
    {
        "id": "doc3_confidentiality",
        "text": (
            "Section 12.1 Confidentiality. The Receiving Party shall maintain all "
            "Confidential Information of the Disclosing Party in strict confidence and shall "
            "not disclose such information to any third party without the prior written "
            "consent of the Disclosing Party, except as required by applicable law or "
            "regulation."
        ),
    },
    {
        "id": "doc4_liability_cap",
        "text": (
            "Section 14.3 Limitation of Liability. In no event shall either party's total "
            "liability under this Agreement exceed the total fees paid by the Client in the "
            "twelve (12) months preceding the event giving rise to the claim, except in cases "
            "of gross negligence, willful misconduct, or breach of the confidentiality "
            "provisions set forth in Section 12."
        ),
    },
    {
        "id": "doc5_governing_law",
        "text": (
            "Section 16.1 Governing Law. This Agreement shall be governed by and construed "
            "in accordance with the laws of the State of Delaware, without regard to its "
            "conflict of laws principles. Any disputes arising hereunder shall be resolved "
            "exclusively in the state or federal courts located in Delaware."
        ),
    },
    {
        "id": "doc6_force_majeure",
        "text": (
            "Section 18.1 Force Majeure. Neither party shall be liable for any delay or "
            "failure to perform its obligations under this Agreement to the extent such delay "
            "or failure results from an event of force majeure, including but not limited to "
            "acts of God, natural disasters, war, or governmental action, provided the "
            "affected party gives prompt written notice."
        ),
    },
]

print(f"Corpus size: {len(legal_documents)} documents")



## Section 4 — Chunking

These sample clauses are already short enough to serve as single chunks. In a real pipeline,
longer documents would be split using one of the strategies from the deck (recursive,
sentence-based, structure-aware). We include a simple recursive-style splitter here so the
pattern is visible even though our sample documents don't strictly need it.


In [ ]:

def simple_chunk(text, max_chars=400, overlap=50):
    # Recursive-ish splitter: break on sentence boundaries, respecting a max chunk size,
    # with a small overlap so information isn't lost at chunk boundaries.
    sentences = text.replace("\n", " ").split(". ")
    chunks, current = [], ""
    for sentence in sentences:
        candidate = (current + ". " + sentence).strip(". ") if current else sentence
        if len(candidate) > max_chars and current:
            chunks.append(current.strip())
            current = current[-overlap:] + " " + sentence
        else:
            current = candidate
    if current:
        chunks.append(current.strip())
    return chunks

# Our sample docs are short, so each becomes exactly one chunk here --
# demonstrated on the longest one to show the function actually splits when needed.
example_chunks = simple_chunk(legal_documents[0]["text"], max_chars=150)
print(f"Example: doc1 split into {len(example_chunks)} chunk(s) at max_chars=150:")
for i, c in enumerate(example_chunks):
    print(f"  Chunk {i}: {c[:80]}...")



## Section 5 — Ingestion: Embed and Store

Each document is embedded and stored in ChromaDB. We use a small, fast open-source embedding
model (`all-MiniLM-L6-v2`) so this runs quickly on CPU with no API cost.


In [ ]:

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

chroma_client = chromadb.Client()  # in-memory, local -- no server, no account
collection = chroma_client.create_collection(name="legal_clauses")

texts = [doc["text"] for doc in legal_documents]
ids = [doc["id"] for doc in legal_documents]
embeddings = embedding_model.encode(texts).tolist()

collection.add(
    ids=ids,
    embeddings=embeddings,
    documents=texts,
)

print(f"Ingested {collection.count()} documents into ChromaDB.")



## Section 6 — Query: Retrieve

Embed the user's question with the SAME embedding model used for ingestion (critical -- see
the deck's warning about this), then run a similarity search.


In [ ]:

def retrieve(query, k=2):
    query_embedding = embedding_model.encode([query]).tolist()
    results = collection.query(query_embeddings=query_embedding, n_results=k)
    return list(zip(results["ids"][0], results["documents"][0], results["distances"][0]))

question = "What happens if the contractor is grossly negligent?"
retrieved = retrieve(question, k=2)

print(f"Question: {question}\n")
print("Retrieved chunks:")
for doc_id, text, distance in retrieved:
    print(f"\n  [{doc_id}] (distance={distance:.4f})")
    print(f"  {text}")



## Section 7 — Query: Generate a Grounded Answer

The retrieved chunks are inserted into the prompt alongside the question -- the model answers
using ONLY what was retrieved, not its own general knowledge of contract law.


In [ ]:

def answer_with_rag(question, k=2):
    retrieved = retrieve(question, k=k)
    context = "\n\n".join(f"[{doc_id}]: {text}" for doc_id, text, _ in retrieved)

    prompt = (
        f"Answer the question using ONLY the context below. Cite the section reference "
        f"if one is present in the context.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}"
    )

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a legal research assistant. Only use the provided context."},
            {"role": "user", "content": prompt},
        ],
        temperature=0,
    )
    return response.choices[0].message.content

print(answer_with_rag("What happens if the contractor is grossly negligent?"))



## Section 8 — Try It Yourself

Ask a few more questions against this small corpus. Notice how the answer stays grounded in
whichever clause actually gets retrieved -- and try a question the corpus can't answer, to see
how the model handles missing context.


In [ ]:

test_questions = [
    "Can either party end this agreement without giving a reason?",
    "What law governs this contract?",
    "What is the maximum liability under this agreement?",
    "What is the penalty for late payment?",  # not covered by our corpus -- watch what happens
]

for q in test_questions:
    print(f"Q: {q}")
    print(f"A: {answer_with_rag(q)}\n")
    print("-" * 70)



## Key Takeaways

1. **The two-phase architecture is now real code**: Sections 3-5 are the ingestion phase (chunk,
   embed, store), Sections 6-7 are the query phase (embed question, retrieve, generate).
2. **Retrieval quality directly bounds answer quality** -- try Section 8's last question and
   notice the model either says it doesn't know, or (if it doesn't) may start relying on general
   knowledge instead of the corpus -- exactly the deck's "retrieval quality is a hard ceiling"
   point.
3. **ChromaDB's local mode required zero setup** -- no server, no account, just a Python
   import -- perfect for prototyping, though a production system would use a persistent,
   scalable vector store.

**Next up:** the *Hybrid Retrieval & Reranking* notebook — adding sparse retrieval and a
reranker on top of this same pipeline.
